# Nepali Image Captioning with BLIP

## Objective

Fine-tune the pretrained Salesforce BLIP image-captioning model to generate
captions in Nepali.

We will investigate:

1. How well pretrained BLIP performs on Nepali without fine-tuning.
2. Whether BLIP's original tokenizer represents Nepali adequately.
3. Whether adding frequent Nepali words to the tokenizer helps.
4. How much fine-tuning improves Nepali caption generation.
5. How the final model performs using BLEU, CIDEr and ROUGE-L.

## Experimental design

Dataset:
- Flickr8k Nepali
- ~8,000 images
- ~40,000 Nepali captions
- approximately 5 captions/image

Split:
- 6,000 images → train
- 1,000 images → validation
- 1,000 images → test

Important:
The split is performed at the IMAGE level, not at the caption level.

This prevents captions belonging to the same image from appearing in
different splits.

Model:
- Salesforce/blip-image-captioning-base

Training:
- pretrained BLIP
- supervised caption fine-tuning
- optional Nepali tokenizer adaptation

Evaluation:
- BLEU-1
- BLEU-2
- BLEU-3
- BLEU-4
- CIDEr
- ROUGE-L

The test set is evaluated only after model selection is complete.

In [ ]:
!pip install -q -U \
    transformers==4.57.1 \
    accelerate \
    sentencepiece \
    rouge-score \
    nltk \
    pycocoevalcap

In [ ]:
import os
import re
import csv
import json
import math
import random
import shutil
import warnings
from pathlib import Path
from collections import Counter, defaultdict

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from PIL import Image
from tqdm.auto import tqdm

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

from pycocoevalcap.cider.cider import Cider
warnings.filterwarnings("ignore")

In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Config

In [ ]:
SEED = 42

MODEL_NAME = "Salesforce/blip-image-captioning-base"

# Dataset split
TRAIN_IMAGES = 6000
VAL_IMAGES = 1000
TEST_IMAGES = 1000

# Training
MAX_EPOCHS = 8          # was 25: too long for a Kaggle session
PATIENCE = 3

BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4   # effective batch = 4 * 4 = 16

LEARNING_RATE = 1e-5        # all weights except the two groups below
EMB_LEARNING_RATE = 1e-4    # word embeddings + LM head (they hold the new Nepali tokens)
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05         # lr rises linearly for the first 5% of steps, then cosine decay

MAX_TEXT_LENGTH = 64

# Generation
NUM_BEAMS = 5
MAX_GENERATION_LENGTH = 40
MIN_GENERATION_LENGTH = 3

# Validation generation
VAL_METRIC_IMAGES = 200

# Tokenizer adaptation
ENABLE_TOKENIZER_ADAPTATION = True
MIN_WORD_FREQUENCY = 2
MAX_NEW_TOKENS = 8000

# DataLoader
NUM_WORKERS = 2

# Smoke test: True = tiny run to check the whole pipeline (set False for the real run)
SMOKE_TEST = False

# Paths
OUTPUT_DIR = Path("/kaggle/working/nepali_blip")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_model_dir = OUTPUT_DIR / "best_model"
LAST_CKPT = OUTPUT_DIR / "last_checkpoint.pt"

print("Output:", OUTPUT_DIR)


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device", DEVICE)

### Find dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bipeshrajsubedi/flickr8k-nepali-dataset")

print("Path to dataset files:", path)

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")

print("Datasets available:")
for p in KAGGLE_INPUT.iterdir():
    print("-", p) 

### Find caption files

In [ ]:
caption_candidates = []

for path in KAGGLE_INPUT.rglob("*"):
    if path.is_file():
        name = path.name.lower()

        if (
            "caption" in name
            and path.suffix.lower() in {".txt", ".csv", ".tsv"}
        ):
            caption_candidates.append(path)

print("Possible caption files:\n")

for p in caption_candidates:
    print(p)

In [ ]:
from pathlib import Path

CAPTION_FILE = Path("/kaggle/input/datasets/bipeshrajsubedi/flickr8k-nepali-dataset/translated_nepali_captions.txt")
IMAGE_DIR = Path("/kaggle/input/datasets/bipeshrajsubedi/flickr8k-nepali-dataset/ficker8k_images/ficker8k_images")

## Caption Parser

1015584366_dfcec3c85a.jpg#0 एउटा कालो कुकुर लग माथि उफ्रन्छ।
1015584366_dfcec3c85a.jpg#1 एउटा खैरो कुकुर ढलेको रूखमाथि उफ्रिरहेको छ।
1015584366_dfcec3c85a.jpg#2 एउटा ठूलो कालो कुकुरले खसेको लग उफ्रन्छ।
1015584366_dfcec3c85a.jpg#3 नीलो कलर लगाएको एउटा कालो र खैरो कुकुर ढलेको रूखमाथि उफ्रिँदै।
1015584366_dfcec3c85a.jpg#4 कालो कुकुरले रूखको टुप्पोमा हाम फाल्यो।
101654506_8eb26cfb60.jpg#0 एउटा खैरो र सेतो कुकुर हिउँमा दौडिरहेको छ।
101654506_8eb26cfb60.jpg#1 एउटा कुकुर हिउँमा दौडिरहेको छ
101654506_8eb26cfb60.jpg#2 हिउँमा दौडिरहेको कुकुर।
101654506_8eb26cfb60.jpg#3 एउटा सेतो र खैरो कुकुर हिउँले ढाकिएको मैदानमा दौडिरहेको छ।
101654506_8eb26cfb60.jpg#4 सेतो र खैरो कुकुर हिउँको सतहमा दौडिरहेको छ।


In [ ]:
def normalize_caption(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def clean_image_id(image_id):
    image_id = str(image_id).strip()

    if "#" in image_id:
        image_id = image_id.split("#")[0]
    return image_id

def parse_caption_file(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line_number, raw_line in enumerate(f, start=1):
            line = raw_line.strip()
            if not line:
                continue

            image_id = None
            caption = None

            # Most Nepali Flickr8k files use tab separation.
            if "\t" in line:
                image_id, caption = line.split("\t", 1)

            # Otherwise try CSV-style format
            elif "," in line:
                try:
                    row = next(csv.reader([line]))

                    if len(row) >= 2:
                        image_id = row[0]
                        caption = ",".join(row[1:])
                except Exception:
                    pass
                    
            # Last fallback: split on first whitespace.
            else:
                parts = line.split(maxsplit=1)

                if len(parts) == 2:
                    image_id, caption = parts

            if image_id is None or caption is None:
                continue

            image_id = clean_image_id(image_id)
            caption = normalize_caption(caption)
            
            # Ignore obvious header rows
            if image_id.lower() in {
                "image",
                "image_name",
                "filename",
                "imageid",
            }:
                continue

            if caption:
                records.append(
                    {
                        "image_id": image_id,
                        "caption": caption,
                    }
                )

    return pd.DataFrame(records)


df = parse_caption_file(CAPTION_FILE)

print("Rows:", len(df))
print("Unique images:", df["image_id"].nunique())

display(df.head())

In [ ]:
# Remove the Nepali full stop (।) and other punctuation from captions BEFORE splitting,
# so "छ।" and "छ" are the same word and metrics are not hurt by attached punctuation.
def clean_nepali_caption(text):
    text = re.sub(r"[।॥,.;:!?\"'“”‘’()\[\]{}\-–—]", " ", str(text))
    return re.sub(r"\s+", " ", text).strip()

df["caption"] = df["caption"].map(clean_nepali_caption)
df = df[df["caption"].str.len() > 0].copy()

print("Rows after cleaning:", len(df))
print(df["caption"].head().tolist())


# Verify the dataset

In [ ]:
caption_counts = df.groupby("image_id").size()

print("Caption count distribution:")
print(caption_counts.value_counts().sort_index())

print("\nMinimum captions/image:", caption_counts.min())
print("Maximum captions/image:", caption_counts.max())

print("\nImages with exactly 5 captions:",
      (caption_counts == 5).sum())

print("\nImages with != 5 captions:",
      (caption_counts != 5).sum())

In [ ]:
image_paths = {}

for path in IMAGE_DIR.rglob("*"):

    if path.is_file() and path.suffix.lower() in {
        ".jpg", ".jpeg", ".png"
    }:
        image_paths[path.name] = path


print("Images discovered:", len(image_paths))

df["image_path"] = df["image_id"].map(image_paths)

missing = df["image_path"].isna()

print("Caption rows with missing images:", missing.sum())

if missing.sum() > 0:
    print("\nExamples:")
    display(df[missing].head())

df = df[~missing].copy()

print("\nRemaining rows:", len(df))
print("Remaining images:", df["image_id"].nunique())

## Image-Level Split

In [ ]:
# ============================================================
# IMAGE-LEVEL SPLIT
# ============================================================
x = df.size
print(x)
all_image_ids = sorted(df["image_id"].unique())

print("Total usable images:", len(all_image_ids))

required_images = TRAIN_IMAGES + VAL_IMAGES + TEST_IMAGES

assert len(all_image_ids) >= required_images, (
    f"Need at least {required_images} images, "
    f"but found {len(all_image_ids)}"
)

rng = random.Random(SEED)
rng.shuffle(all_image_ids)

train_ids = all_image_ids[:TRAIN_IMAGES]

val_ids = all_image_ids[
    TRAIN_IMAGES:
    TRAIN_IMAGES + VAL_IMAGES
]

test_ids = all_image_ids[
    TRAIN_IMAGES + VAL_IMAGES:
    TRAIN_IMAGES + VAL_IMAGES + TEST_IMAGES
]

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

### split dataframes

In [ ]:
train_df = df[df["image_id"].isin(train_ids)].copy()
val_df = df[df["image_id"].isin(val_ids)].copy()
test_df = df[df["image_id"].isin(test_ids)].copy()

print("Train caption rows:", len(train_df))
print("Val caption rows:", len(val_df))
print("Test caption rows:", len(test_df))

## verify no leakage

In [ ]:
train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

print("Train ∩ Val:", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Val ∩ Test:", len(val_set & test_set))

assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0

print("\nNo image-level leakage detected.")

In [ ]:
split_info = {
    "seed": SEED,
    "train_images": train_ids,
    "val_images": val_ids,
    "test_images": test_ids,
}

with open(OUTPUT_DIR / "splits.json", "w", encoding="utf-8") as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_DIR / "splits.json")

In [ ]:
# Smoke test: shrink everything so the whole notebook runs end to end in minutes.
if SMOKE_TEST:
    train_ids = train_ids[:40]
    val_ids   = val_ids[:10]
    test_ids  = test_ids[:10]
    train_df = train_df[train_df.image_id.isin(train_ids)].copy()
    val_df   = val_df[val_df.image_id.isin(val_ids)].copy()
    test_df  = test_df[test_df.image_id.isin(test_ids)].copy()
    VAL_METRIC_IMAGES = 10
    MAX_EPOCHS = 1
    # separate files, so the real run never resumes from the smoke-test checkpoint
    LAST_CKPT = OUTPUT_DIR / "smoke_last.pt"
    best_model_dir = OUTPUT_DIR / "smoke_best"
    print("SMOKE TEST:", len(train_df), len(val_df), len(test_df))


In [ ]:
def build_references(dataframe):
    references = defaultdict(list)
    
    for _, row in dataframe.iterrows():
        references[row["image_id"]].append(row["caption"])

    return dict(references)
    
train_references = build_references(train_df)
val_references = build_references(val_df)
test_references = build_references(test_df)

print("Train reference images:", len(train_references))
print("Val reference images:", len(val_references))
print("Test reference images:", len(test_references))

example_id = list(train_references.keys())[0]

print("\nExample:")
print(example_id)

for caption in train_references[example_id]:
    print(" -", caption)

# LOAD BLIP

Image  -> BlipProcessor -> Pixel Values -> BLIP model 

In [ ]:
from transformers import BertTokenizerFast

# Load BLIP's image preprocessing + tokenizer.
processor = BlipProcessor.from_pretrained(MODEL_NAME)

# BLIP's tokenizer is English BERT-uncased: by default it STRIPS combining marks
# (Devanagari vowel signs / viramas), turning कुकुर into क ##क ##र.
# strip_accents=False keeps them.
processor.tokenizer = BertTokenizerFast.from_pretrained(
    MODEL_NAME, strip_accents=False
)

# Load the pretrained BLIP image-captioning model and weights.
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(DEVICE)

print("Model loaded.")
print("Tokenizer size:", len(processor.tokenizer))
print("Model vocab size:", model.config.text_config.vocab_size)
print("bos_token_id (decoder start):", model.config.text_config.bos_token_id)
print(processor.tokenizer.tokenize("एउटा कुकुर दौडिरहेको छ"))


#### understanding tokenizer

In [ ]:
sample_captions = train_df["caption"].sample(
    min(10, len(train_df)),
    random_state=SEED
).tolist()

for caption in sample_captions:
    tokens = processor.tokenizer.tokenize(caption)
    print("\nCAPTION:")
    print(caption)

    print("TOKENS:")
    print(tokens)

Now our concern is of UNK present in token. 


In [ ]:
total_tokens = 0
unk_tokens = 0

for caption in train_df["caption"]:
    tokens = processor.tokenizer.tokenize(caption)

    total_tokens += len(tokens)
    unk_tokens += tokens.count("[UNK]")

print("Total tokens:", total_tokens)
print("UNK tokens:", unk_tokens)
print("UNK rate:", unk_tokens / total_tokens)

In [ ]:
UNK_TOKEN = processor.tokenizer.unk_token

print("UNK token:", UNK_TOKEN)
print("UNK token id:", processor.tokenizer.unk_token_id)

measure unknown token rate

In [ ]:
def tokenizer_statistics(captions):
    total_tokens = 0
    unknown_tokens = 0
    captions_with_unknown = 0
    
    token_lengths = []

    for caption in captions:
        ids = processor.tokenizer(
            caption,
            add_special_tokens=True
        )["input_ids"]

        total_tokens += len(ids)

        unknown_count = sum(
            token_id == processor.tokenizer.unk_token_id
            for token_id in ids
        )

        unknown_tokens += unknown_count

        if unknown_count > 0:
            captions_with_unknown += 1

        token_lengths.append(len(ids))
    return {
        "total_tokens": total_tokens,
        "unknown_tokens": unknown_tokens,
        "unknown_token_rate": (
            unknown_tokens / total_tokens
            if total_tokens > 0 else 0
        ),
        "captions_with_unknown": captions_with_unknown,
        "caption_unknown_rate": (
            captions_with_unknown / len(captions)
            if captions else 0
        ),
        "average_token_length": np.mean(token_lengths),
    }

token_stats = tokenizer_statistics(
    train_df["caption"].tolist(),
)

token_stats

## Baseline dataset

In [ ]:
class CaptionDataset(Dataset):
    def __init__(
        self,
        dataframe,
        processor,
        max_length=64,
        bos_token_id=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length
        self.bos_token_id = bos_token_id

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        caption = row["caption"]

        encoded = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {key: value.squeeze(0) for key,value in encoded.items()}

        # BLIP generates starting from its decoder token ([DEC], id = bos_token_id),
        # so train with the same first token instead of [CLS].
        if self.bos_token_id is not None:
            item["input_ids"][0] = self.bos_token_id

        # BLIP's loss ignores -100 labels.
        labels = item["input_ids"].clone()

        labels[
            labels == self.processor.tokenizer.pad_token_id
        ] = -100

        item["labels"] = labels

        return item

        

## Baseline interference

Before training anything, we want to know what the pretrained model can already do

In [ ]:
@torch.no_grad()
def generate_caption(image, model, processor, num_beams=5, max_length=40, min_length=3):
    model.eval()

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(DEVICE) for k,v in inputs.items()
    }


    generated_ids = model.generate(
        **inputs,
        num_beams=num_beams,
        max_length=max_length,
        min_length=min_length,
        early_stopping=True,
        no_repeat_ngram_size=2,
    )

    caption = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return caption.strip()

baseline qualitative examples

In [ ]:
sample_test_ids = random.sample(
    test_ids,
    min(10, len(test_ids))
)


baseline_examples = []

for image_id in tqdm(sample_test_ids):
    image_path = image_paths[image_id]
    image = Image.open(image_path).convert("RGB")

    prediction = generate_caption(
        image,
        model,
        processor,
    )

    baseline_examples.append({
        "image_id": image_id,
        "prediction": prediction,
        "references": test_references[image_id],
    })

In [ ]:
from IPython.display import display
from PIL import Image

for example in baseline_examples:

    print("=" * 80)

    print("IMAGE:", example["image_id"])

    image_path = image_paths[example["image_id"]]
    image = Image.open(image_path).convert("RGB")
    display(image)

    print("\nBASELINE:")
    print(example["prediction"])

    print("\nREFERENCES:")

    for ref in example["references"]:
        print(" -", ref)

# Extract Nepali vocabulary

In [ ]:
DEVANAGARI_WORD = re.compile(
    r"^[\u0900-\u097F]+$"
)

def build_nepali_vocabulary(
    captions,
    min_frequency=2,
    max_tokens=8000,
):
    counter = Counter()

    for caption in captions:
        words = caption.split()
        
        for word in words:
            # Remove punctuation around words.
            cleaned = re.sub(
                r"[^\u0900-\u097F]",
                "",
                word
            )
            if not cleaned:
                continue

            if DEVANAGARI_WORD.match(cleaned):
                counter[cleaned] += 1

    vocabulary = [
        word
        for word, count in counter.most_common()
        if count >= min_frequency
    ]

    return vocabulary[:max_tokens], counter


nepali_vocab, word_counter = build_nepali_vocabulary(
    train_df["caption"].tolist(),
    min_frequency=MIN_WORD_FREQUENCY,
    max_tokens=MAX_NEW_TOKENS,
)

print("Candidate Nepali words:", len(nepali_vocab))

print("\nTop 50:")
print(nepali_vocab[:50])



### compare tokenization before modification

In [ ]:
def tokenize_words_with_original_tokenizer(
    captions,
    tokenizer,
):
    results = []
    for caption in captions:
        words = caption.split()
        for word in words:
            cleaned = re.sub(
                r"[^\u0900-\u097F]",
                "",
                word
            )

            if not cleaned:
                continue

            tokens = tokenizer.tokenize(cleaned)
            results.append({
                "word": cleaned,
                "tokens": tokens,
            })
    return results


tokenization_examples = tokenize_words_with_original_tokenizer(
    sample_captions,
    processor.tokenizer,
)

for item in tokenization_examples[:30]:
    print(f"{item['word']:20s} -> {item['tokens']}")

## Apply tokenizer adaptation

In [ ]:
from transformers import AddedToken

original_vocab_size = len(processor.tokenizer)

# The model has 2 more embedding rows (30524) than the tokenizer (30522): ids 30522
# ([DEC], the decoder start token) and 30523 belong to BLIP. Reserve them so that new
# Nepali tokens do NOT get these ids.
n_reserved = model.config.text_config.vocab_size - len(processor.tokenizer)
if n_reserved > 0:
    reserved = ["[DEC]", "[ENC]"][:n_reserved]
    processor.tokenizer.add_special_tokens({"additional_special_tokens": reserved})
    assert processor.tokenizer.convert_tokens_to_ids("[DEC]") == model.config.text_config.bos_token_id, \
        "[DEC] id does not match bos_token_id"

if ENABLE_TOKENIZER_ADAPTATION:
    vocab = processor.tokenizer.get_vocab()
    new_tokens = [
        AddedToken(w, single_word=True, normalized=True)
        for w in nepali_vocab if w not in vocab
    ]
    print("Original vocabulary:", original_vocab_size)
    print("Candidate new tokens:", len(new_tokens))
    num_added = processor.tokenizer.add_tokens(new_tokens)
    print("Actually added:", num_added)
else:
    num_added = 0
    print("Tokenizer adaptation disabled.")

if len(processor.tokenizer) != model.get_input_embeddings().num_embeddings:
    model.resize_token_embeddings(len(processor.tokenizer))

print("New vocabulary size:", len(processor.tokenizer))

# Round-trip check: decoded text should equal the original caption
for c in train_df["caption"].sample(5, random_state=1):
    ids = processor.tokenizer(c)["input_ids"]
    print(c, "\n ->", processor.tokenizer.decode(ids, skip_special_tokens=True), "\n")


In [ ]:
# ============================================================
# VERIFY TOKENIZER / MODEL VOCABULARY ALIGNMENT
# ============================================================

tokenizer_size = len(processor.tokenizer)

# Input embedding vocabulary
input_embeddings = model.get_input_embeddings()

assert input_embeddings is not None, (
    "Model does not have an input embedding layer."
)

input_embedding_size = input_embeddings.num_embeddings

print("Tokenizer vocabulary:", tokenizer_size)
print("Model input embeddings:", input_embedding_size)

# ------------------------------------------------------------
# Check tokenizer ↔ input embedding alignment
# ------------------------------------------------------------

assert tokenizer_size == input_embedding_size, (
    f"Tokenizer size ({tokenizer_size}) does not match "
    f"model input embedding size ({input_embedding_size})!"
)

# ------------------------------------------------------------
# Inspect output/language-model head
# ------------------------------------------------------------

output_embeddings = model.get_output_embeddings()

if output_embeddings is not None:
    output_embedding_size = output_embeddings.out_features

    print("Model output embeddings:", output_embedding_size)

    assert tokenizer_size == output_embedding_size, (
        f"Tokenizer size ({tokenizer_size}) does not match "
        f"model output embedding size ({output_embedding_size})!"
    )
else:
    print(
        "Model output embeddings: None "
        "(BLIP does not expose a separate output embedding "
        "through get_output_embeddings())"
    )

print()
print("✓ Tokenizer and model vocabulary sizes are aligned.")

measure tokenizer again

In [ ]:
adapted_stats = tokenizer_statistics(
    train_df["caption"].tolist()
)

print("Before:")
print(token_stats)

print("\nAfter:")
print(adapted_stats)

print("\nComparison:")
print(
    "UNK rate:",
    token_stats["unknown_token_rate"],
    "->",
    adapted_stats["unknown_token_rate"]
)

print(
    "Average token length:",
    token_stats["average_token_length"],
    "->",
    adapted_stats["average_token_length"]
)

In [ ]:
# ============================================================
# CHECK CAPTION TOKEN LENGTHS
# ============================================================

caption_lengths = []

for caption in train_df["caption"]:

    tokenized = processor.tokenizer(
        caption,
        add_special_tokens=True,
        truncation=False,
    )

    caption_lengths.append(
        len(tokenized["input_ids"])
    )

caption_lengths = np.array(
    caption_lengths
)

print(
    "Maximum token length:",
    caption_lengths.max(),
)

print(
    "Mean token length:",
    caption_lengths.mean(),
)

print(
    "95th percentile:",
    np.percentile(
        caption_lengths,
        95,
    ),
)

print(
    "99th percentile:",
    np.percentile(
        caption_lengths,
        99,
    ),
)

num_truncated = np.sum(
    caption_lengths > MAX_TEXT_LENGTH
)

truncation_percentage = (
    100
    * num_truncated
    / len(caption_lengths)
)

print()
print(
    f"Captions longer than "
    f"MAX_TEXT_LENGTH ({MAX_TEXT_LENGTH}): "
    f"{num_truncated}"
)

print(
    f"Percentage that would be truncated: "
    f"{truncation_percentage:.2f}%"
)

## Build final datasets

In [ ]:
train_dataset = CaptionDataset(
    train_df,
    processor,
    max_length=MAX_TEXT_LENGTH,
    bos_token_id=model.config.text_config.bos_token_id,
)

val_dataset = CaptionDataset(
    val_df,
    processor,
    max_length=MAX_TEXT_LENGTH,
    bos_token_id=model.config.text_config.bos_token_id,
)

test_dataset = CaptionDataset(
    test_df,
    processor,
    max_length=MAX_TEXT_LENGTH,
    bos_token_id=model.config.text_config.bos_token_id,
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

# Smoke Test

In [ ]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print("key: ", key)
    print("shape", value.shape)
    print("type", value.dtype)


# Forward pass

In [ ]:
model.eval()

with torch.no_grad():
    batch_gpu = {
        k: v.to(DEVICE) for k,v  in batch.items()
    }
    outputs = model(
        **batch_gpu
    )

print("Loss:", outputs.loss.item())
print("Logits:", outputs.logits.shape)

# validation loss

In [ ]:
def evaluate_loss(model, loader):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch in tqdm(
            loader, desc="Validation",
            leave = False,
        ):
            batch = {
                k: v.to(DEVICE)
                for k, v in batch.items()
            }

            outputs = model(**batch)

            total_loss += outputs.loss.item()
            total_batches += 1
    return total_loss / total_batches

# Validation caption metrics

In [ ]:
val_metric_ids = sorted(val_ids)[:VAL_METRIC_IMAGES]
val_metric_references = {
    image_id: val_references[image_id]
    for image_id in val_metric_ids
}

print(
    "Validation metric subset:",
    len(val_metric_ids)
)

# Metric functions

In [ ]:
smooth = SmoothingFunction().method4


def normalize_metric_text(text):
    """
    Nepali-friendly tokenization.

    We intentionally use whitespace tokenization rather than
    English-specific linguistic preprocessing.
    """
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)

    return text.split()


def calculate_bleu(
    references,
    predictions,
):

    refs = []
    hyps = []

    for image_id, prediction in predictions.items():

        reference_tokens = [
            normalize_metric_text(x)
            for x in references[image_id]
        ]

        prediction_tokens = normalize_metric_text(
            prediction
        )

        refs.append(reference_tokens)
        hyps.append(prediction_tokens)

    scores = {}

    for n, weights in {
        1: (1.0, 0, 0, 0),
        2: (0.5, 0.5, 0, 0),
        3: (1/3, 1/3, 1/3, 0),
        4: (0.25, 0.25, 0.25, 0.25),
    }.items():

        scores[f"BLEU-{n}"] = corpus_bleu(
            refs,
            hyps,
            weights=weights,
            smoothing_function=smooth,
        )

    return scores

# ROUGE - L

In [ ]:
class WhitespaceTokenizer:
    """rouge_score's default tokenizer deletes all non-ASCII text, so use whitespace."""
    def tokenize(self, text):
        return str(text).split()

rouge = rouge_scorer.RougeScorer(["rougeL"], tokenizer=WhitespaceTokenizer())

def calculate_rouge_l(
    references,
    predictions,
):

    scores = []

    for image_id, prediction in predictions.items():

        best_score = 0.0

        for reference in references[image_id]:

            score = rouge.score(
                reference,
                prediction
            )["rougeL"].fmeasure

            best_score = max(
                best_score,
                score
            )

        scores.append(best_score)

    return float(np.mean(scores))

In [1]:
import pandas as pd, numpy as np
from rouge_score import rouge_scorer

class WhitespaceTokenizer:
    def tokenize(self, text):
        return str(text).split()

scorer = rouge_scorer.RougeScorer(["rougeL"], tokenizer=WhitespaceTokenizer())
preds = pd.read_csv("/kaggle/working/nepali_blip/test_predictions.csv")

scores = []
for _, r in preds.iterrows():
    refs = r["references"].split(" || ")
    scores.append(max(scorer.score(ref, r["prediction"])["rougeL"].fmeasure for ref in refs))

print("Test ROUGE-L:", round(float(np.mean(scores)), 4))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/nepali_blip/test_predictions.csv'

# CIDEr

In [ ]:
def calculate_cider(references, predictions):

    gts = {}
    res = {}

    for index, (image_id, prediction) in enumerate(
        predictions.items()
    ):
        gts[index] = [
            " ".join(normalize_metric_text(ref))
            for ref in references[image_id]
        ]

        res[index] = [
            " ".join(normalize_metric_text(prediction))
        ]

    cider = Cider()

    score, individual_scores = cider.compute_score(
        gts,
        res
    )

    return float(score)

# Batch caption generation

In [ ]:
@torch.no_grad()
def generate_predictions(model, processor, image_ids, references=None):
    model.eval()

    predictions = {}

    for image_id in tqdm(image_ids, desc="Generating captions", leave=False):
        image = Image.open(image_paths[image_id]).convert("RGB")
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        generated_ids = model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_length=MAX_GENERATION_LENGTH,
            min_length=MIN_GENERATION_LENGTH,
            early_stopping=True,
            no_repeat_ngram_size=2,
            use_cache=True,   # model.config.use_cache is False for training; generation is slow without it
        )
        predictions[image_id] = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0].strip()

    return predictions


# Validation metrics

In [ ]:
def evaluate_generation_metrics(
    model,
    processor,
    image_ids,
    references,
):

    predictions = generate_predictions(
        model,
        processor,
        image_ids,
        references,
    )

    bleu_scores = calculate_bleu(
        references,
        predictions
    )

    rouge_l = calculate_rouge_l(
        references,
        predictions
    )

    cider = calculate_cider(
        references,
        predictions
    )

    metrics = {
        **bleu_scores,
        "ROUGE-L": rouge_l,
        "CIDEr": cider,
    }

    return metrics, predictions

# Optimizer

In [ ]:
emb_params, other_params = [], []
for n, p in model.named_parameters():
    if "word_embeddings" in n or "text_decoder.cls" in n:
        emb_params.append(p)
    else:
        other_params.append(p)
print("embedding/head params:", sum(p.numel() for p in emb_params),
      "| other params:", sum(p.numel() for p in other_params))

optimizer = torch.optim.AdamW(
    [
        {"params": other_params, "lr": LEARNING_RATE},
        {"params": emb_params, "lr": EMB_LEARNING_RATE},
    ],
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(
    len(train_loader) /
    GRADIENT_ACCUMULATION_STEPS
)

total_training_steps = (
    steps_per_epoch *
    MAX_EPOCHS
)

warmup_steps = int(
    total_training_steps *
    WARMUP_RATIO
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print("Steps/epoch:", steps_per_epoch)
print("Total optimizer steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

# Gradient Checkpointing

In [ ]:
model.gradient_checkpointing_enable()

model.config.use_cache=False
print("Gradient checkpointing enabled")

# Training Loop

In [ ]:
# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    scaler,
    epoch,
):
    model.train()

    total_loss = 0.0

    progress = tqdm(
        enumerate(loader),
        total=len(loader),
        desc=f"Epoch {epoch}",
    )

    optimizer.zero_grad(set_to_none=True)

    for step, batch in progress:

        # Move batch to device
        batch = {
            key: value.to(DEVICE)
            if torch.is_tensor(value)
            else value
            for key, value in batch.items()
        }

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            outputs = model(**batch)

            original_loss = outputs.loss

            # Determine how many batches belong to this
            # gradient accumulation group.
            group_start = (
                step // GRADIENT_ACCUMULATION_STEPS
            ) * GRADIENT_ACCUMULATION_STEPS

            group_size = min(
                GRADIENT_ACCUMULATION_STEPS,
                len(loader) - group_start,
            )

            loss = original_loss / group_size
            

        # ----------------------------------------------------
        # Backward pass
        # ----------------------------------------------------
        scaler.scale(loss).backward()

        # Record the ORIGINAL loss from every batch.
        total_loss += original_loss.item()

        # ----------------------------------------------------
        # Optimizer step
        # ----------------------------------------------------
        should_update = (
            (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            or step == len(loader) - 1
        )

        if should_update:

            # Unscale before gradient clipping.
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        
        # ----------------------------------------------------
        # Progress display
        # ----------------------------------------------------
        progress.set_postfix(
            loss=f"{original_loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}",
        )

    # Average over ALL batches, not only optimizer steps.
    average_loss = total_loss / len(loader)

    return average_loss

# TRaining + Validation

In [ ]:
# ============================================================
# MIXED PRECISION GRADIENT SCALER
# ============================================================

import torch

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

print(
    "AMP GradScaler enabled:",
    torch.cuda.is_available()
)

In [ ]:
# ============================================================
# TRAINING
# ============================================================

history = {k: [] for k in
           ["epoch", "train_loss", "val_loss", "bleu1", "bleu2", "bleu3", "bleu4", "rouge_l", "cider"]}

best_metric = -float("inf")
best_epoch = -1
patience_counter = 0
start_epoch = 1

# Resume if a previous run left a checkpoint in the same working directory
if LAST_CKPT.exists():
    ck = torch.load(LAST_CKPT, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model"])
    model.to(DEVICE)
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    scaler.load_state_dict(ck["scaler"])
    history = ck["history"]
    best_metric = ck["best_metric"]
    best_epoch = ck["best_epoch"]
    patience_counter = ck["patience_counter"]
    start_epoch = ck["epoch"] + 1
    print("Resumed after epoch", ck["epoch"])

val_metric_ids = sorted(val_ids)[:VAL_METRIC_IMAGES]
val_metric_references = {i: val_references[i] for i in val_metric_ids}

for epoch in range(start_epoch, MAX_EPOCHS + 1):

    print()
    print("=" * 70)
    print(f"Epoch {epoch}/{MAX_EPOCHS}")
    print("=" * 70)

    model.config.use_cache = False
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        epoch=epoch,
    )
    print("Train loss for epoch", train_loss)

    model.config.use_cache = True
    val_loss = evaluate_loss(model, val_loader)

    print()
    print("Generating validation captions...")
    metrics, val_predictions = evaluate_generation_metrics(
        model=model,
        processor=processor,
        image_ids=val_metric_ids,
        references=val_metric_references,
    )

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["bleu1"].append(metrics["BLEU-1"])
    history["bleu2"].append(metrics["BLEU-2"])
    history["bleu3"].append(metrics["BLEU-3"])
    history["bleu4"].append(metrics["BLEU-4"])
    history["rouge_l"].append(metrics["ROUGE-L"])
    history["cider"].append(metrics["CIDEr"])

    print()
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"BLEU-1     : {metrics['BLEU-1']:.4f}")
    print(f"BLEU-2     : {metrics['BLEU-2']:.4f}")
    print(f"BLEU-3     : {metrics['BLEU-3']:.4f}")
    print(f"BLEU-4     : {metrics['BLEU-4']:.4f}")
    print(f"ROUGE-L    : {metrics['ROUGE-L']:.4f}")
    print(f"CIDEr      : {metrics['CIDEr']:.4f}")
    k = next(iter(val_predictions))
    print("Sample:", val_predictions[k])

    if metrics["CIDEr"] > best_metric:
        best_metric = metrics["CIDEr"]
        best_epoch = epoch
        patience_counter = 0

        print(f"\n✓ New best CIDEr: {best_metric:.4f}")
        if best_model_dir.exists():
            shutil.rmtree(best_model_dir)
        best_model_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(best_model_dir)
        processor.save_pretrained(best_model_dir)
        print("✓ Best model saved to:", best_model_dir)
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")

    # Resumable checkpoint after every epoch
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "history": history,
        "best_metric": best_metric,
        "best_epoch": best_epoch,
        "patience_counter": patience_counter,
        "epoch": epoch,
    }, LAST_CKPT)

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print("Best epoch:", best_epoch)
print("Best CIDEr:", best_metric)
print("Best model:", best_model_dir)


# Interpret the training curves

In [ ]:
history_df = pd.DataFrame(history)

display(history_df)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
    label="Train loss"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    marker="o",
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

# Final test evaluation

In [ ]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================
# Reload the BEST checkpoint chosen by validation CIDEr (not the last epoch).

print("Loading best checkpoint from:", best_model_dir)

model = BlipForConditionalGeneration.from_pretrained(best_model_dir)
processor = BlipProcessor.from_pretrained(best_model_dir)
# make sure the reloaded tokenizer still keeps Devanagari vowel signs
processor.tokenizer = BertTokenizerFast.from_pretrained(best_model_dir, strip_accents=False)
model.to(DEVICE)
model.eval()
model.config.use_cache = True

print("✓ Best checkpoint loaded (epoch", best_epoch, ")")
print(processor.tokenizer.tokenize("एउटा कुकुर दौडिरहेको छ"))

test_predictions = generate_predictions(model, processor, test_ids, test_references)
print("Generated captions for", len(test_predictions), "test images.")


# Final metrics

In [ ]:
test_bleu = calculate_bleu(
    test_references,
    test_predictions
)

test_rouge = calculate_rouge_l(
    test_references,
    test_predictions
)

test_cider = calculate_cider(
    test_references,
    test_predictions
)

final_metrics = {
    **test_bleu,
    "ROUGE-L": test_rouge,
    "CIDEr": test_cider,
}

print("\nFINAL TEST RESULTS")
print("=" * 50)

for key, value in final_metrics.items():
    print(f"{key:15s}: {value:.6f}")

# Save predictions

In [ ]:
prediction_rows = []

for image_id, prediction in test_predictions.items():

    prediction_rows.append({
        "image_id": image_id,
        "prediction": prediction,
        "references": " || ".join(
            test_references[image_id]
        ),
    })


predictions_df = pd.DataFrame(
    prediction_rows
)

predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False,
    encoding="utf-8"
)

print(
    "Saved:",
    OUTPUT_DIR / "test_predictions.csv"
)

# Save experiment info

In [ ]:
experiment_config = {
    "model": MODEL_NAME,

    "dataset": {
        "train_images": TRAIN_IMAGES,
        "validation_images": VAL_IMAGES,
        "test_images": TEST_IMAGES,
    },

    "training": {
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
    },

    "generation": {
        "num_beams": NUM_BEAMS,
        "max_length": MAX_GENERATION_LENGTH,
        "min_length": MIN_GENERATION_LENGTH,
    },

    "tokenizer": {
        "original_vocab_size": original_vocab_size,
        "new_tokens_added": num_added,
        "final_vocab_size": len(processor.tokenizer),
        "before_statistics": token_stats,
        "after_statistics": adapted_stats,
    },

    "best_epoch": best_epoch,

    "validation_history": history,

    "test_metrics": final_metrics,
}


with open(
    OUTPUT_DIR / "experiment.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_config,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Saved:",
    OUTPUT_DIR / "experiment.json"
)

# Qualitative evaluation

In [ ]:
qualitative_ids = random.sample(
    test_ids,
    min(15, len(test_ids))
)

for image_id in qualitative_ids:

    image = Image.open(
        image_paths[image_id]
    ).convert("RGB")

    display(image.resize((384, 384)))

    print("\nIMAGE:", image_id)

    print("\nGENERATED:")
    print(test_predictions[image_id])

    print("\nREFERENCES:")

    for reference in test_references[image_id]:
        print(" -", reference)

    print("\n" + "=" * 100)

# Compare baseline vs fine-tuned

In [ ]:
# Generate baseline predictions on the SAME test set.
# This is still useful because it gives us an actual
# pretrained-vs-fine-tuned comparison.

baseline_model = BlipForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

baseline_processor = BlipProcessor.from_pretrained(
    MODEL_NAME
)

baseline_model.to(DEVICE)
baseline_model.eval()

baseline_test_predictions = {}

for image_id in tqdm(
    test_ids,
    desc="Baseline test generation"
):

    image = Image.open(
        image_paths[image_id]
    ).convert("RGB")

    inputs = baseline_processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        generated_ids = baseline_model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_length=MAX_GENERATION_LENGTH,
            min_length=MIN_GENERATION_LENGTH,
            early_stopping=True,
            no_repeat_ngram_size=2,
        )

    prediction = baseline_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    baseline_test_predictions[image_id] = prediction

# Baseline metrics

In [ ]:
baseline_bleu = calculate_bleu(
    test_references,
    baseline_test_predictions
)

baseline_rouge = calculate_rouge_l(
    test_references,
    baseline_test_predictions
)

baseline_cider = calculate_cider(
    test_references,
    baseline_test_predictions
)

baseline_metrics = {
    **baseline_bleu,
    "ROUGE-L": baseline_rouge,
    "CIDEr": baseline_cider,
}

print("BASELINE")
print("=" * 50)

for key, value in baseline_metrics.items():
    print(f"{key:15s}: {value:.6f}")

# Experimental comparision

In [ ]:
comparison = pd.DataFrame(
    {
        "Pretrained BLIP": baseline_metrics,
        "Fine-tuned BLIP": final_metrics,
    }
)

comparison

# Save everything

In [ ]:
history_df.to_csv(
    OUTPUT_DIR / "training_history.csv",
    index=False
)

with open(
    OUTPUT_DIR / "baseline_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        baseline_metrics,
        f,
        indent=2
    )

with open(
    OUTPUT_DIR / "final_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_metrics,
        f,
        indent=2
    )

print("Experiment artifacts saved to:")
print(OUTPUT_DIR)

# Inference function for your final project

In [ ]:
def caption_nepali_image(
    image_path,
    model=model,
    processor=processor,
):

    image = Image.open(
        image_path
    ).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    model.eval()

    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_length=MAX_GENERATION_LENGTH,
            min_length=MIN_GENERATION_LENGTH,
            early_stopping=True,
            no_repeat_ngram_size=2,
        )

    caption = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return caption.strip()

# Download model 

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/best_model", "zip", "/kaggle/working/nepali_blip/best_model")

In [2]:
!ls -laR /kaggle/working/nepali_blip | head -50
!du -sh /kaggle/working/nepali_blip/*

/kaggle/working/nepali_blip:
total 268
drwxr-xr-x 2 root root   4096 Oct  5 14:42 .
drwxr-xr-x 4 root root   4096 Oct  5 14:44 ..
-rw-r--r-- 1 root root 262346 Oct  5 14:42 splits.json
260K	/kaggle/working/nepali_blip/splits.json
